# Image Classification Using Transfer Learning Demonstration In Python

## Background
Image classification is a fundamental task in deep learning where the goal is
to assign a label to an image from a fixed set of categories.


Training deep neural networks from scratch requires:
- Large datasets
- High computational power


To overcome this, we use **Transfer Learning:**
- Use a pretrained model (trained on ImageNet)
- Adapt it to a new task (CIFAR-10)


**Here, we use pretrained ResNet/VGG models and fine-tune only the final layer to make the training efficient and fast on CPU.**


## What is the CIFAR-10 Dataset?
- A large collection of 60,000 color images (RGB), each of size 32×32 pixels.
- The dataset is divided into 10 object categories: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.
- Data split:

   • 50,000 images for training
   
   • 10,000 images for testing

## Process Flow 
1. Load CIFAR-10 dataset
2. Use pretrained CNN (ResNet/VGG)
3. Freeze most layers (feature extraction)
4. Train only the final classification layer
5. Evaluate performance on test data


### Import Libraries

torch: Core PyTorch library used for building, training, and running deep learning models

torchvision: Provides datasets, pretrained models, and image transformations for computer vision tasks

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import time
import random
import numpy as np

### Reproducibility
Sets a fixed random seed to ensure reproducible results across runs.

In [4]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

### Device Configuration

Automatically selects the best available hardware (GPU if possible, else CPU) for training.

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


### Data Preprocessing

- Resize → Makes all images same size (needed for neural networks)
- ToTensor → Converts images into numbers PyTorch understands
- Normalize → Standardizes pixel values → improves model training
- Trainset vs Testset
     1. Train → used to learn
     2. Test → used to evaluate
- DataLoader
    1. Breaks dataset into batches
    2. Enables faster training
    3. Shuffle helps avoid learning bias

**Note:**  Although pretrained models such as **ResNet-18 / VGG-16 (trained on ImageNet)** typically expect 224×224 inputs, we use 64×64 images to reduce computational overhead and speed up training, which is suitable for quick experimentation on CIFAR-10.

In [7]:
# Mean and standard deviation values for normalization
# These are precomputed for ImageNet dataset (used by ResNet)
# Helps in faster convergence and better performance
stats = ((0.485, 0.456, 0.406),   # Mean for R, G, B channels
         (0.229, 0.224, 0.225))   # Std deviation for R, G, B channels


# Define transformations to be applied on each image
transform = transforms.Compose([
    
    # Resize all images to 64x64
    # CIFAR-10 images are originally 32x32, but we upscale for ResNet
    transforms.Resize((64, 64)),  
    
    # Convert image (PIL format) to PyTorch tensor
    # Also scales pixel values from [0, 255] → [0, 1]
    transforms.ToTensor(),
    
    # Normalize images using mean and std
    # Formula: (image - mean) / std
    # Makes training stable and improves learning
    transforms.Normalize(*stats)
])


# Load CIFAR-10 training dataset
# - root: where dataset will be stored
# - train=True: training data
# - download=True: downloads if not already present
# - transform: apply preprocessing defined above
trainset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)


# Load CIFAR-10 test dataset
testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)


# DataLoader for training data
# - batch_size=128: number of images processed at once
# - shuffle=True: randomizes data each epoch (important for training)
# - num_workers=2: parallel loading for speed
trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=128,
    shuffle=True,
    num_workers=2
)


# DataLoader for test data
# - shuffle=False: keep order fixed for evaluation
testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

### Model Initialization

- Uses a pretrained model (ResNet/VGG) → already knows basic image features
- Freezes all layers → no retraining of existing knowledge
- Replaces last layer → adjusts from 1000 classes → 10 classes
- Trains only final layer → adapts model to your dataset

In [9]:
# Function to initialize model (ResNet or VGG)
def initialize_model(model_name="resnet", num_classes=10):
    
    # RESNET MODEL
    if model_name == "resnet":
        
        # Load pretrained ResNet18 model (trained on ImageNet)
        # This means model already knows general image features (edges, shapes, textures)
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        # Freeze all layers
         #This "turns off" backpropagation for those layers to preserve the pre-trained weights
        # We do not want to retrain the entire network (saves time & avoids overfitting)
        for param in model.parameters():
            param.requires_grad = False
        
        # Replace final fully connected (FC) layer
        # Original FC layer is for 1000 classes (ImageNet)
        # We replace it with our own layer for CIFAR-10 (10 classes)
        num_ftrs = model.fc.in_features   # Number of input features to FC layer
        model.fc = nn.Linear(num_ftrs, num_classes)

    # VGG MODEL
    elif model_name == "vgg":
        
        # Load pretrained VGG16 model with batch normalization
        model = models.vgg16_bn(weights=models.VGG16_Bn_Weights.DEFAULT)
        
        # Freeze all layers 
        #This "turns off" backpropagation for those layers to preserve the pre-trained weights

        for param in model.parameters():
            param.requires_grad = False
        
        # Replace final classifier layer
        # VGG has a classifier block (not a single fc like ResNet)
        num_ftrs = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(num_ftrs, num_classes)

    # Move model to device (CPU or GPU)
    return model.to(device)


# Initialize model (ResNet by default)
model = initialize_model("resnet")

### Loss And Optimizer

- Loss → measures how wrong the model is
- Optimizer → updates model weights (only last layer here)
- Scheduler → reduces learning rate over time

In [10]:
# Loss function for classification
# CrossEntropyLoss is used when we have multiple classes (like CIFAR-10)
criterion = nn.CrossEntropyLoss()


# Optimizer: Stochastic Gradient Descent (SGD)
# Only updates parameters where requires_grad=True (i.e., last layer)
optimizer = optim.SGD(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.01,          # learning rate (step size)
    momentum=0.9      # helps accelerate training
)


# Learning Rate Scheduler
# Reduces learning rate after every 2 epochs
# gamma=0.1 → LR becomes 10% of previous value
scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=2,
    gamma=0.1
)

### Train Model

Train the model by predicting, computing error, and updating weights repeatedly.

In [11]:
def train_model(model, trainloader, criterion, optimizer, scheduler, epochs=3):
    
    # Set model to training mode (important for dropout, batchnorm, etc.)
    model.train()

    # Loop over epochs
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        start_time = time.time()   # track time per epoch

        # Loop over batches
        for inputs, labels in trainloader:
            
            # Move data to device (CPU/GPU)
            inputs, labels = inputs.to(device), labels.to(device)

            # Reset gradients (important before each batch)
            optimizer.zero_grad()
            
            # Forward pass → get predictions
            outputs = model(inputs)
            
            # Compute loss (error)
            loss = criterion(outputs, labels)

            # Backward pass → compute gradients
            loss.backward()
            
            # Update weights (only last layer)
            optimizer.step()

            # Accumulate loss
            running_loss += loss.item()

            # Get predicted class (max probability)
            _, predicted = torch.max(outputs, 1)

            # Update accuracy metrics
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        # Step the scheduler (reduce learning rate if needed)
        scheduler.step()

        # Print epoch results
        print(f"Epoch {epoch+1}: Loss={running_loss/len(trainloader):.4f}, "
              f"Accuracy={100*correct/total:.2f}%, "
              f"Time={time.time()-start_time:.2f}s")

### Evaluation 
Check how well the trained model performs on unseen data.

In [13]:
def evaluate_model(model, testloader):
    
    # Set model to evaluation mode
    # Disables dropout, uses running stats for batchnorm
    model.eval()
    
    correct = 0
    total = 0

    # Disable gradient computation (saves memory & speeds up)
    with torch.no_grad():
        for inputs, labels in testloader:
            
            # Move data to device (CPU/GPU)
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass → get predictions
            outputs = model(inputs)
            
            # Get predicted class (highest probability)
            _, predicted = torch.max(outputs, 1)

            # Update accuracy metrics
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    # Print final test accuracy
    print(f"Test Accuracy: {100*correct/total:.2f}%")

### Run

**Note** : You can increase the number of epochs to improve accuracy,as the model gets more chances to learn from the data. However, too many epochs may lead to overfitting.

In [14]:
# Train the model for 3 epochs
# This will update only the final layer weights
train_model(model, trainloader, criterion, optimizer, scheduler, epochs=3)

# Evaluate the trained model on test data
# Gives final accuracy on unseen data
evaluate_model(model, testloader)

Epoch 1: Loss=1.2185, Accuracy=59.60%, Time=661.51s
Epoch 2: Loss=1.1192, Accuracy=62.53%, Time=666.91s
Epoch 3: Loss=0.9994, Accuracy=65.87%, Time=666.38s
Test Accuracy: 65.36%


### Interpretation :
The model is steadily learning (loss decreasing, accuracy increasing) and achieves a reasonable ~65% performance on unseen data after 3 epochs.